# Lesson 26 — Differentiable Optimization

## Learning objectives

1. Differentiate an *argmin* with respect to its parameters via the implicit-function theorem.
2. Use OptNet / cvxpylayers in a neural network {cite:p}`Amos2017,Agrawal2019`.
3. Recognize the contexts where differentiable optimization wins.
4. Understand the link to decision-focused learning (Lesson 27).

## 1. The implicit-function theorem

For an optimization problem $x^\star(\theta) = \arg\min_x f(x, \theta)$, suppose $\nabla_x f(x^\star, \theta) = 0$. Differentiating implicitly:

$$\nabla_{xx}^2 f \cdot \frac{\partial x^\star}{\partial \theta} + \nabla_{x \theta}^2 f = 0,$$

so

$$\frac{\partial x^\star}{\partial \theta} = -[\nabla_{xx}^2 f]^{-1} \nabla_{x \theta}^2 f.$$

Same idea works for KKT systems with active inequality constraints — differentiate the KKT system implicitly {cite:p}`Amos2017`.

## 2. OptNet

Embed a QP layer

$$z(\theta) = \arg\min_z \tfrac{1}{2} z^\top Q(\theta) z + p(\theta)^\top z \;\;\text{s.t.}\;\; A(\theta) z = b(\theta), \; G(\theta) z \le h(\theta)$$

inside a neural network. The forward pass solves the QP; the backward pass differentiates through the KKT system. {cite:p}`Amos2017`.

For convex programs in DCP form, `cvxpylayers` {cite:p}`Agrawal2019` automates this — define your CP, get a PyTorch / JAX layer.

## 3. When differentiable optimization wins

- **Constraint as inductive bias:** the network output is *guaranteed* to satisfy a physical constraint (e.g., $\sum_i w_i = 1$, a probability).
- **Decision-focused learning:** train upstream prediction to minimize downstream decision regret (Lesson 27).
- **Differentiable physics:** PDEs as optimization layers.

When *not* to use it: when the constraint is a soft preference (just penalize), when the QP is large (forward + backward both expensive), or when discrete constraints dominate (differentiating MILP is hard).

## 4. Differentiating MIP

Strict integer constraints are non-differentiable; approximations:

- **LP relaxation + STE:** use LP-relaxed gradient on the backward pass.
- **Smoothed perturbed argmax:** add Gumbel noise, take softmax {cite:p}`Mandi2020`.
- **Sub-gradient via Fenchel-Young loss** {cite:p}`Elmachtoub2022`.

These give *useful* but biased gradients.

### Working tools for differentiable convex programs

`discopt` does not ship a differentiable-optimization layer. For exercises
2 and 3 below, install **`cvxpylayers`** (PyTorch / JAX) or **`qpth`**
(the OptNet QP solver). A typical workflow:

```python
import cvxpy as cp
from cvxpylayers.torch import CvxpyLayer

mu_p = cp.Parameter(n_assets)
w_v  = cp.Variable(n_assets)
prob = cp.Problem(
    cp.Minimize(0.5 * cp.quad_form(w_v, Sigma) - mu_p @ w_v),
    [cp.sum(w_v) == 1, w_v >= 0],
)
layer = CvxpyLayer(prob, parameters=[mu_p], variables=[w_v])
# layer(torch_mu) is now differentiable w.r.t. torch_mu.
```

The decision-focused exercise below (E3) gives you the loss-function pattern;
the convex layer is what makes the gradient meaningful.

## 5. Caveats

- Forward pass: solver call per example per batch.
- Backward pass: extra solve / KKT linear system.
- Numerical conditioning: differentiating near the boundary of the feasible region or near degeneracy is unstable.

## References
{cite:p}`Amos2017,Agrawal2019,Mandi2020,Elmachtoub2022`.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm
